# 05_04 - CNN-BiLSTM Sequence (seed 123)

Train CNN-BiLSTM-Attention on real token sequences with behavioral late fusion. This notebook trains **one seed only** (`SEED = 123`) to avoid long Colab runs.

Sibling notebooks: `05_04a` seed 42, `05_04b` seed 123, `05_04c` seed 456. Index: `05_04_CNN_BiLSTM_Sequence.ipynb`.


## Contract

- Run in Colab only. Read Phase 2 raw 777 artifacts.
- Fixed seed: **123**. Outputs go to canonical paths for seed 42, else `artifacts/*/seed_123/`.
- Write probabilities, metrics and metadata for downstream `05_05` multi-seed blending.
- File: `05_04b_CNN_BiLSTM_Sequence_seed123.ipynb`


In [2]:
# try/except: khối xử lý ngoại lệ
try:
    # import google.colab  # type: ignore: import thư viện google
    import google.colab  # type: ignore
    # IN_COLAB: biến cấu hình/hằng số của notebook
    IN_COLAB = True
# except: xử lý ngoại lệ — except ImportError:
except ImportError:
    # IN_COLAB: biến cấu hình/hằng số của notebook
    IN_COLAB = False

# if: điều kiện — if not IN_COLAB:
if not IN_COLAB:
    # raise RuntimeError("Run this notebook in Google Colab. Do not execute Phase 5 tr...: ném lỗi và dừng cell
    raise RuntimeError("Run this notebook in Google Colab. Do not execute Phase 5 training locally.")


In [3]:
# from google.colab import drive: import thư viện google
from google.colab import drive
# drive.mount("/content/drive"): mount Google Drive trên Colab
drive.mount("/content/drive")


Mounted at /content/drive


In [4]:
# import gc: giải phóng bộ nhớ
import gc
# import importlib: import thư viện importlib
import importlib
# import json: đọc/ghi JSON metadata
import json
# import os: biến môi trường hệ thống
import os
# import random: cố định seed ngẫu nhiên
import random
# import subprocess: chạy lệnh pip/cài package
import subprocess
# import sys: tham số Python runtime
import sys
# import time: đo thời gian thực thi
import time
# from datetime import datetime, timezone: import thư viện datetime
from datetime import datetime, timezone
# from itertools import combinations, product: import thư viện itertools
from itertools import combinations, product
# from pathlib import Path: quản lý đường dẫn
from pathlib import Path

# import joblib: lưu/tải object đã fit
import joblib
# import numpy: tính toán mảng số
import numpy as np
# import pandas: xử lý DataFrame
import pandas as pd
# from sklearn.metrics import (: thư viện machine learning scikit-learn
from sklearn.metrics import (
    # accuracy_score,: thực thi lệnh Python
    accuracy_score,
    # average_precision_score,: thực thi lệnh Python
    average_precision_score,
    # brier_score_loss,: thực thi lệnh Python
    brier_score_loss,
    # confusion_matrix,: thực thi lệnh Python
    confusion_matrix,
    # f1_score,: thực thi lệnh Python
    f1_score,
    # precision_recall_fscore_support,: thực thi lệnh Python
    precision_recall_fscore_support,
    # roc_auc_score,: thực thi lệnh Python
    roc_auc_score,
# ): đóng ngoặc gọi hàm
)

# SEED: biến cấu hình/hằng số của notebook
SEED = 123
# FAKE_LABEL: biến cấu hình/hằng số của notebook
FAKE_LABEL = 1
# REAL_LABEL: biến cấu hình/hằng số của notebook
REAL_LABEL = 0
# DEFAULT_THRESHOLD: biến cấu hình/hằng số của notebook
DEFAULT_THRESHOLD = 0.50
# TARGET_PRECISION_FAKE: biến cấu hình/hằng số của notebook
TARGET_PRECISION_FAKE = 0.975

# PROJECT_ROOT: biến cấu hình/hằng số của notebook
PROJECT_ROOT = Path(os.environ.get("FAKE_REVIEWS_PROJECT_ROOT", "/content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews"))
# FEATURE_DIR: biến cấu hình/hằng số của notebook
FEATURE_DIR = PROJECT_ROOT / "artifacts" / "features"
# MODEL_DIR: biến cấu hình/hằng số của notebook
MODEL_DIR = PROJECT_ROOT / "artifacts" / "models"
# ENSEMBLE_DIR: biến cấu hình/hằng số của notebook
ENSEMBLE_DIR = PROJECT_ROOT / "artifacts" / "ensemble"
# PREDICTION_DIR: biến cấu hình/hằng số của notebook
PREDICTION_DIR = PROJECT_ROOT / "artifacts" / "predictions"
# REPORT_TABLE_DIR: biến cấu hình/hằng số của notebook
REPORT_TABLE_DIR = PROJECT_ROOT / "reports" / "tables"
# REPORT_FIGURE_DIR: biến cấu hình/hằng số của notebook
REPORT_FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
# PROCESSED_DIR: biến cấu hình/hằng số của notebook
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# for: vòng lặp — for directory in [REPORT_FIGURE_DIR]:
for directory in [REPORT_FIGURE_DIR]:
    # directory.mkdir(parents=True, exist_ok=True): tạo thư mục nếu chưa có
    directory.mkdir(parents=True, exist_ok=True)

# RAW_FEATURE_PATHS: biến cấu hình/hằng số của notebook
RAW_FEATURE_PATHS = {
    # "train": FEATURE_DIR / "features_raw_train.npy",: thực thi lệnh Python
    "train": FEATURE_DIR / "features_raw_train.npy",
    # "val": FEATURE_DIR / "features_raw_val.npy",: thực thi lệnh Python
    "val": FEATURE_DIR / "features_raw_val.npy",
    # "test": FEATURE_DIR / "features_raw_test.npy",: thực thi lệnh Python
    "test": FEATURE_DIR / "features_raw_test.npy",
# }: đóng khối từ điển
}
# LABEL_PATHS: biến cấu hình/hằng số của notebook
LABEL_PATHS = {
    # "train": FEATURE_DIR / "labels_train.npy",: thực thi lệnh Python
    "train": FEATURE_DIR / "labels_train.npy",
    # "val": FEATURE_DIR / "labels_val.npy",: thực thi lệnh Python
    "val": FEATURE_DIR / "labels_val.npy",
    # "test": FEATURE_DIR / "labels_test.npy",: thực thi lệnh Python
    "test": FEATURE_DIR / "labels_test.npy",
# }: đóng khối từ điển
}
# FEATURE_METADATA_PATH: biến cấu hình/hằng số của notebook
FEATURE_METADATA_PATH = FEATURE_DIR / "feature_metadata.json"


# configure_seed_artifacts: thiết lập đường dẫn artifact theo seed
def configure_seed_artifacts(seed: int) -> int:
    # """Canonical paths for seed 42; isolated subfolders for other seeds.""": thực thi lệnh Python
    """Canonical paths for seed 42; isolated subfolders for other seeds."""
    # global SEED, PREDICTION_DIR, ENSEMBLE_DIR, REPORT_TABLE_DIR, MODEL_DIR: thực thi lệnh Python
    global SEED, PREDICTION_DIR, ENSEMBLE_DIR, REPORT_TABLE_DIR, MODEL_DIR
    # SEED: biến cấu hình/hằng số của notebook
    SEED = int(seed)
    # if: điều kiện — if SEED == 42:
    if SEED == 42:
        # prediction_dir = ...: dự đoán nhãn/xác suất
        prediction_dir = PROJECT_ROOT / "artifacts" / "predictions"
        # ensemble_dir = ...: gán giá trị cho biến ensemble dir
        ensemble_dir = PROJECT_ROOT / "artifacts" / "ensemble"
        # report_table_dir = ...: gán giá trị cho biến report table dir
        report_table_dir = PROJECT_ROOT / "reports" / "tables"
        # model_dir = ...: gán giá trị cho biến model dir
        model_dir = PROJECT_ROOT / "artifacts" / "models"
    # else: nhánh còn lại của điều kiện
    else:
        # tag = ...: gán giá trị cho biến tag
        tag = f"seed_{SEED}"
        # prediction_dir = ...: dự đoán nhãn/xác suất
        prediction_dir = PROJECT_ROOT / "artifacts" / "predictions" / tag
        # ensemble_dir = ...: gán giá trị cho biến ensemble dir
        ensemble_dir = PROJECT_ROOT / "artifacts" / "ensemble" / tag
        # report_table_dir = ...: gán giá trị cho biến report table dir
        report_table_dir = PROJECT_ROOT / "reports" / "tables" / tag
        # model_dir = ...: gán giá trị cho biến model dir
        model_dir = PROJECT_ROOT / "artifacts" / "models" / tag
    # for: vòng lặp — for directory in [prediction_dir, ensemble_dir, report_table
    for directory in [prediction_dir, ensemble_dir, report_table_dir, model_dir]:
        # directory.mkdir(parents=True, exist_ok=True): tạo thư mục nếu chưa có
        directory.mkdir(parents=True, exist_ok=True)
    # PREDICTION_DIR: biến cấu hình/hằng số của notebook
    PREDICTION_DIR = prediction_dir
    # ENSEMBLE_DIR: biến cấu hình/hằng số của notebook
    ENSEMBLE_DIR = ensemble_dir
    # REPORT_TABLE_DIR: biến cấu hình/hằng số của notebook
    REPORT_TABLE_DIR = report_table_dir
    # MODEL_DIR: biến cấu hình/hằng số của notebook
    MODEL_DIR = model_dir
    # return: trả kết quả từ hàm
    return SEED


# set_global_seed: cố định seed random và numpy
def set_global_seed(seed: int) -> None:
    # random.seed(seed): cố định seed random
    random.seed(seed)
    # np.random.seed(seed): cố định seed numpy
    np.random.seed(seed)


# set_torch_seed: hàm xử lý set torch seed
def set_torch_seed(seed: int) -> None:
    # import torch: deep learning PyTorch
    import torch

    # torch.manual_seed(seed): cố định seed torch
    torch.manual_seed(seed)
    # if: điều kiện — if torch.cuda.is_available():
    if torch.cuda.is_available():
        # torch.cuda.manual_seed_all(seed): thực thi lệnh Python
        torch.cuda.manual_seed_all(seed)


# configure_seed_artifacts(SEED): thực thi lệnh Python
configure_seed_artifacts(SEED)
# set_global_seed(SEED): tạo tập hợp
set_global_seed(SEED)
# set_torch_seed(SEED): tạo tập hợp
set_torch_seed(SEED)


# utc_now: định nghĩa hàm utc now
def utc_now() -> str:
    # return: trả kết quả từ hàm
    return datetime.now(timezone.utc).isoformat()


# ensure_package: import hoặc pip install package
def ensure_package(import_name: str, pip_name: str | None = None):
    # try/except: khối xử lý ngoại lệ
    try:
        # return: trả kết quả từ hàm
        return importlib.import_module(import_name)
    # except: xử lý ngoại lệ — except ImportError:
    except ImportError:
        # subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name or impor...: thực thi lệnh Python
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name or import_name])
        # return: trả kết quả từ hàm
        return importlib.import_module(import_name)


# read_json: đọc file JSON
def read_json(path: Path, default=None):
    # if: điều kiện — if not path.exists():
    if not path.exists():
        # return: trả kết quả từ hàm
        return default if default is not None else {}
    # with: context manager — with path.open("r", encoding="utf-8") as file:
    with path.open("r", encoding="utf-8") as file:
        # return: parse nội dung JSON
        return json.load(file)


# load_raw_arrays: nạp feature/label .npy từ Phase 2
def load_raw_arrays():
    # missing = ...: kiểm tra file/thư mục tồn tại
    missing = [str(path) for path in list(RAW_FEATURE_PATHS.values()) + list(LABEL_PATHS.values()) if not path.exists()]
    # if: điều kiện — if missing:
    if missing:
        # raise FileNotFoundError("Missing Phase 2 raw feature inputs: " + json.dumps(miss...: ghi dictionary ra JSON
        raise FileNotFoundError("Missing Phase 2 raw feature inputs: " + json.dumps(missing, indent=2))
    # X: biến cấu hình/hằng số của notebook
    X = {split: np.load(path).astype(np.float32, copy=False) for split, path in RAW_FEATURE_PATHS.items()}
    # y = ...: nạp mảng từ file .npy
    y = {split: np.load(path).astype(np.int64, copy=False) for split, path in LABEL_PATHS.items()}
    # for: vòng lặp — for split in ["train", "val", "test"]:
    for split in ["train", "val", "test"]:
        # if: điều kiện — if X[split].ndim != 2:
        if X[split].ndim != 2:
            # raise ValueError(f"{split} raw features must be 2D, got {X[split].shape}"): ném lỗi và dừng cell
            raise ValueError(f"{split} raw features must be 2D, got {X[split].shape}")
        # if: điều kiện — if y[split].ndim != 1 or len(y[split]) != X[split].shape[0]:
        if y[split].ndim != 1 or len(y[split]) != X[split].shape[0]:
            # raise ValueError(f"{split} label/features mismatch: X={X[split].shape}, y={y[spl...: ném lỗi và dừng cell
            raise ValueError(f"{split} label/features mismatch: X={X[split].shape}, y={y[split].shape}")
        # if: điều kiện — if not np.isfinite(X[split]).all():
        if not np.isfinite(X[split]).all():
            # raise ValueError(f"Non-finite raw features in {split}"): ném lỗi và dừng cell
            raise ValueError(f"Non-finite raw features in {split}")
    # return: trả kết quả từ hàm
    return X, y


# safe_roc_auc: ROC-AUC an toàn
def safe_roc_auc(y_true, prob_fake):
    # try/except: khối xử lý ngoại lệ
    try:
        # return: trả kết quả từ hàm
        return float(roc_auc_score(y_true, prob_fake))
    # except: xử lý ngoại lệ — except ValueError:
    except ValueError:
        # return: trả kết quả từ hàm
        return float("nan")


# safe_pr_auc: PR-AUC an toàn
def safe_pr_auc(y_true, prob_fake):
    # try/except: khối xử lý ngoại lệ
    try:
        # return: trả kết quả từ hàm
        return float(average_precision_score(y_true, prob_fake, pos_label=FAKE_LABEL))
    # except: xử lý ngoại lệ — except ValueError:
    except ValueError:
        # return: trả kết quả từ hàm
        return float("nan")


# evaluate_predictions: tính metric classification từ xác suất
def evaluate_predictions(y_true, prob_fake, split, model_variant, threshold=DEFAULT_THRESHOLD, threshold_strategy="default_0.5"):
    # y_true = ...: ép kiểu dữ liệu cột
    y_true = np.asarray(y_true).astype(int)
    # prob_fake = ...: ép kiểu dữ liệu cột
    prob_fake = np.asarray(prob_fake).astype(float)
    # y_pred = ...: ép kiểu dữ liệu cột
    y_pred = (prob_fake >= threshold).astype(int)
    # precision, recall, f1, support = precision_recall_fscore_support(: thực thi lệnh Python
    precision, recall, f1, support = precision_recall_fscore_support(
        # y_true, y_pred, labels=[REAL_LABEL, FAKE_LABEL], zero_division=0: thực thi lệnh Python
        y_true, y_pred, labels=[REAL_LABEL, FAKE_LABEL], zero_division=0
    # ): đóng ngoặc gọi hàm
    )
    # tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[REAL_LABEL, FAKE_LABEL...: thực thi lệnh Python
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[REAL_LABEL, FAKE_LABEL]).ravel()
    # return: trả kết quả từ hàm
    return {
        # "generated_at_utc": utc_now(),: thực thi lệnh Python
        "generated_at_utc": utc_now(),
        # "seed": int(SEED),: ép kiểu số nguyên
        "seed": int(SEED),
        # "split": split,: thực thi lệnh Python
        "split": split,
        # "model_variant": model_variant,: thực thi lệnh Python
        "model_variant": model_variant,
        # "threshold": float(threshold),: ép kiểu số thực
        "threshold": float(threshold),
        # "threshold_strategy": threshold_strategy,: ép kiểu chuỗi
        "threshold_strategy": threshold_strategy,
        # "accuracy": float(accuracy_score(y_true, y_pred)),: ép kiểu số thực
        "accuracy": float(accuracy_score(y_true, y_pred)),
        # "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),: ép kiểu số thực
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        # "precision_fake": float(precision[1]),: ép kiểu số thực
        "precision_fake": float(precision[1]),
        # "recall_fake": float(recall[1]),: ép kiểu số thực
        "recall_fake": float(recall[1]),
        # "f1_fake": float(f1[1]),: ép kiểu số thực
        "f1_fake": float(f1[1]),
        # "support_real": int(support[0]),: ép kiểu số nguyên
        "support_real": int(support[0]),
        # "support_fake": int(support[1]),: ép kiểu số nguyên
        "support_fake": int(support[1]),
        # "roc_auc": safe_roc_auc(y_true, prob_fake),: thực thi lệnh Python
        "roc_auc": safe_roc_auc(y_true, prob_fake),
        # "pr_auc": safe_pr_auc(y_true, prob_fake),: thực thi lệnh Python
        "pr_auc": safe_pr_auc(y_true, prob_fake),
        # "brier_score": float(brier_score_loss(y_true, prob_fake)),: ép kiểu số thực
        "brier_score": float(brier_score_loss(y_true, prob_fake)),
        # "tn": int(tn),: ép kiểu số nguyên
        "tn": int(tn),
        # "fp": int(fp),: ép kiểu số nguyên
        "fp": int(fp),
        # "fn": int(fn),: ép kiểu số nguyên
        "fn": int(fn),
        # "tp": int(tp),: ép kiểu số nguyên
        "tp": int(tp),
    # }: đóng khối từ điển
    }


# save_probability: lưu xác suất fake ra file .npy
def save_probability(prob_fake, model_variant, split):
    # path = ...: gán giá trị cho biến path
    path = PREDICTION_DIR / f"{model_variant}_{split}_prob.npy"
    # np.save(path, np.asarray(prob_fake, dtype=np.float32)): lưu mảng numpy ra file .npy
    np.save(path, np.asarray(prob_fake, dtype=np.float32))
    # return: trả kết quả từ hàm
    return str(path)


# write_metrics: ghi bảng metric ra CSV
def write_metrics(prob_map, y, model_variant, output_name):
    # rows = ...: gán giá trị cho biến rows
    rows = []
    # for: vòng lặp — for split in ["train", "val", "test"]:
    for split in ["train", "val", "test"]:
        # row = ...: dự đoán nhãn/xác suất
        row = evaluate_predictions(y[split], prob_map[split], split=split, model_variant=model_variant)
        # row["probability_path"] = save_probability(prob_map[split], model_variant, split...: thực thi lệnh Python
        row["probability_path"] = save_probability(prob_map[split], model_variant, split)
        # rows.append(row): thực thi lệnh Python
        rows.append(row)
    # df = ...: gán giá trị cho biến df
    df = pd.DataFrame(rows)
    # path = ...: gán giá trị cho biến path
    path = REPORT_TABLE_DIR / output_name
    # df.to_csv(path, index=False): ghi DataFrame ra file CSV
    df.to_csv(path, index=False)
    # display(df): hiển thị bảng/kết quả trên notebook
    display(df)
    # print("Saved metrics:", path): in thông tin ra console
    print("Saved metrics:", path)
    # return: trả kết quả từ hàm
    return df, path


In [5]:
# import torch: deep learning PyTorch
import torch
# import torch.nn as nn: deep learning PyTorch
import torch.nn as nn
# from torch.utils.data import DataLoader, Dataset: deep learning PyTorch
from torch.utils.data import DataLoader, Dataset
# from sklearn.preprocessing import StandardScaler: thư viện machine learning scikit-learn
from sklearn.preprocessing import StandardScaler
# transformers = ...: transform dữ liệu bằng object đã fit
transformers = ensure_package("transformers")
# from transformers import AutoTokenizer: HuggingFace transformers trong pipeline dự án
from transformers import AutoTokenizer

# BERT_MODEL_NAME: biến cấu hình/hằng số của notebook
BERT_MODEL_NAME = "answerdotai/ModernBERT-base"
# MAX_LENGTH: biến cấu hình/hằng số của notebook
MAX_LENGTH = 160
# BATCH_SIZE: biến cấu hình/hằng số của notebook
BATCH_SIZE = 64
# EMBED_DIM: biến cấu hình/hằng số của notebook
EMBED_DIM = 256
# DEVICE: biến cấu hình/hằng số của notebook
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [6]:
# infer_columns: hàm xử lý infer columns
def infer_columns(frame, schema):
    # text_col = ...: gán giá trị cho biến text col
    text_col = schema.get("text_col") or schema.get("text") or next((c for c in ["text", "review", "reviewText", "review_text"] if c in frame.columns), None)
    # label_col = ...: gán giá trị cho biến label col
    label_col = schema.get("label_col") or schema.get("label") or next((c for c in ["label", "class", "target"] if c in frame.columns), None)
    # if: điều kiện — if text_col is None or label_col is None:
    if text_col is None or label_col is None:
        # raise ValueError(f"Could not infer columns from {frame.columns.tolist()}"): ném lỗi và dừng cell
        raise ValueError(f"Could not infer columns from {frame.columns.tolist()}")
    # return: trả kết quả từ hàm
    return text_col, label_col

# schema = ...: đọc file JSON
schema = read_json(PROCESSED_DIR / "schema_metadata.json", default={})
# frames = ...: đọc file CSV vào DataFrame
frames = {split: pd.read_csv(PROCESSED_DIR / f"{split}.csv") for split in ["train", "val", "test"]}
# TEXT_COL, LABEL_COL = infer_columns(frames["train"], schema): thực thi lệnh Python
TEXT_COL, LABEL_COL = infer_columns(frames["train"], schema)
# behavior_paths = ...: gán giá trị cho biến behavior paths
behavior_paths = {split: FEATURE_DIR / f"behavioral_{split}.csv" for split in ["train", "val", "test"]}
# behavior_frames = ...: đọc file CSV vào DataFrame
behavior_frames = {split: pd.read_csv(behavior_paths[split]) for split in ["train", "val", "test"]}
# behavior_cols = ...: gán giá trị cho biến behavior cols
behavior_cols = [c for c in behavior_frames["train"].columns if c != "row_id"]
# scaler = ...: gán giá trị cho biến scaler
scaler = StandardScaler()
# behavior = ...: fit scaler trên train và transform
behavior = {"train": scaler.fit_transform(behavior_frames["train"][behavior_cols]).astype(np.float32), "val": scaler.transform(behavior_frames["val"][behavior_cols]).astype(np.float32), "test": scaler.transform(behavior_frames["test"][behavior_cols]).astype(np.float32)}
# y = ...: ép kiểu dữ liệu cột
y = {split: frames[split][LABEL_COL].astype(int).to_numpy(dtype=np.int64) for split in ["train", "val", "test"]}
# joblib.dump(scaler, ENSEMBLE_DIR / "phase5_cnn_bilstm_behavior_scaler.joblib"): lưu object (scaler/model) ra disk
joblib.dump(scaler, ENSEMBLE_DIR / "phase5_cnn_bilstm_behavior_scaler.joblib")
# print("TEXT_COL:", TEXT_COL, "LABEL_COL:", LABEL_COL, "behavior_dim:", len(behav...: in thông tin ra console
print("TEXT_COL:", TEXT_COL, "LABEL_COL:", LABEL_COL, "behavior_dim:", len(behavior_cols))


TEXT_COL: text LABEL_COL: label behavior_dim: 9


In [7]:
# tokenizer = ...: gán giá trị cho biến tokenizer
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
# vocab_size = ...: đếm số phần tử
vocab_size = int(tokenizer.vocab_size or len(tokenizer))
# pad_token_id = ...: gán giá trị cho biến pad token id
pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
# encoded = ...: gán giá trị cho biến encoded
encoded = {}
# for: vòng lặp — for split in ["train", "val", "test"]:
for split in ["train", "val", "test"]:
    # encoded[split] = tokenizer(frames[split][TEXT_COL].fillna("").astype(str).tolist...: điền giá trị thay thế cho NaN
    encoded[split] = tokenizer(frames[split][TEXT_COL].fillna("").astype(str).tolist(), padding="max_length", truncation=True, max_length=MAX_LENGTH, return_tensors="np")

# class SequenceDataset: định nghĩa lớp
class SequenceDataset(Dataset):
    # __init__: hàm xử lý   init  
    def __init__(self, split):
        # self.input_ids = torch.tensor(encoded[split]["input_ids"], dtype=torch.long): thực thi lệnh Python
        self.input_ids = torch.tensor(encoded[split]["input_ids"], dtype=torch.long)
        # self.mask = torch.tensor(encoded[split]["attention_mask"], dtype=torch.float32): ép kiểu số thực
        self.mask = torch.tensor(encoded[split]["attention_mask"], dtype=torch.float32)
        # self.behavior = torch.tensor(behavior[split], dtype=torch.float32): ép kiểu số thực
        self.behavior = torch.tensor(behavior[split], dtype=torch.float32)
        # self.labels = torch.tensor(y[split], dtype=torch.float32): ép kiểu số thực
        self.labels = torch.tensor(y[split], dtype=torch.float32)
    # __len__: hàm xử lý   len  
    def __len__(self): return len(self.labels)
    # __getitem__: hàm xử lý   getitem  
    def __getitem__(self, idx): return self.input_ids[idx], self.mask[idx], self.behavior[idx], self.labels[idx]

# make_loader: tạo PyTorch DataLoader
def make_loader(split, shuffle=False):
    # return: trả kết quả từ hàm
    return DataLoader(SequenceDataset(split), batch_size=BATCH_SIZE, shuffle=shuffle, pin_memory=torch.cuda.is_available())

# class AttentionPooling: định nghĩa lớp
class AttentionPooling(nn.Module):
    # __init__: hàm xử lý   init  
    def __init__(self, input_dim, attention_dim=128):
        # super().__init__(): thực thi lệnh Python
        super().__init__()
        # self.score = nn.Sequential(nn.Linear(input_dim, attention_dim), nn.Tanh(), nn.Li...: thực thi lệnh Python
        self.score = nn.Sequential(nn.Linear(input_dim, attention_dim), nn.Tanh(), nn.Linear(attention_dim, 1))
    # forward: hàm xử lý forward
    def forward(self, sequence_output, mask):
        # scores = ...: gán giá trị cho biến scores
        scores = self.score(sequence_output).squeeze(-1).masked_fill(mask <= 0, -1e4)
        # weights = ...: lấy giá trị lớn nhất
        weights = torch.softmax(scores, dim=1)
        # return: trả kết quả từ hàm
        return torch.sum(sequence_output * weights.unsqueeze(-1), dim=1)

# class CNNBiLSTMSequenceLateFusion: định nghĩa lớp
class CNNBiLSTMSequenceLateFusion(nn.Module):
    # __init__: hàm xử lý   init  
    def __init__(self, vocab_size, behavior_dim):
        # super().__init__(): thực thi lệnh Python
        super().__init__()
        # self.embedding = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=pad_token_id): thực thi lệnh Python
        self.embedding = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=pad_token_id)
        # self.conv = nn.Sequential(nn.Conv1d(EMBED_DIM, 128, 3, padding=1), nn.BatchNorm1...: thực thi lệnh Python
        self.conv = nn.Sequential(nn.Conv1d(EMBED_DIM, 128, 3, padding=1), nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(0.30))
        # self.lstm = nn.LSTM(128, 96, batch_first=True, bidirectional=True): thực thi lệnh Python
        self.lstm = nn.LSTM(128, 96, batch_first=True, bidirectional=True)
        # self.attention = AttentionPooling(192): thực thi lệnh Python
        self.attention = AttentionPooling(192)
        # self.behavior_branch = nn.Sequential(nn.Linear(behavior_dim, 32), nn.GELU(), nn....: thực thi lệnh Python
        self.behavior_branch = nn.Sequential(nn.Linear(behavior_dim, 32), nn.GELU(), nn.Dropout(0.30))
        # self.classifier = nn.Sequential(nn.Linear(192 + 32, 96), nn.GELU(), nn.Dropout(0...: thực thi lệnh Python
        self.classifier = nn.Sequential(nn.Linear(192 + 32, 96), nn.GELU(), nn.Dropout(0.30), nn.Linear(96, 1))
    # forward: hàm xử lý forward
    def forward(self, input_ids, mask, behavior):
        # x = ...: gán giá trị cho biến x
        x = self.embedding(input_ids).transpose(1, 2)
        # x = ...: gán giá trị cho biến x
        x = self.conv(x).transpose(1, 2)
        # x, _ = self.lstm(x): thực thi lệnh Python
        x, _ = self.lstm(x)
        # return: trả kết quả từ hàm
        return self.classifier(torch.cat([self.attention(x, mask), self.behavior_branch(behavior)], dim=1)).squeeze(-1)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

In [8]:
# configure_seed_artifacts(SEED): thực thi lệnh Python
configure_seed_artifacts(SEED)
# set_global_seed(SEED): tạo tập hợp
set_global_seed(SEED)
# set_torch_seed(SEED): tạo tập hợp
set_torch_seed(SEED)
# print(f"\n=== seed={SEED} ==="): in thông tin ra console
print(f"\n=== seed={SEED} ===")
# MODEL_VARIANT: biến cấu hình/hằng số của notebook
MODEL_VARIANT = "phase5_cnn_bilstm_sequence"
# CONFIG: biến cấu hình/hằng số của notebook
CONFIG = {"model_name": BERT_MODEL_NAME, "max_length": MAX_LENGTH, "batch_size": BATCH_SIZE, "max_epochs": 20, "patience": 5}
# vocab_size = ...: lấy giá trị lớn nhất
vocab_size = max(vocab_size, pad_token_id + 1)
# model = ...: xóa biến để giải phóng RAM/VRAM
model = CNNBiLSTMSequenceLateFusion(vocab_size, len(behavior_cols)).to(DEVICE)
# criterion = ...: gán giá trị cho biến criterion
criterion = nn.BCEWithLogitsLoss()
# optimizer = ...: gán giá trị cho biến optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=8e-4, weight_decay=1e-4)

# @torch.no_grad(): thực thi lệnh Python
@torch.no_grad()
# predict: hàm xử lý predict
def predict(split):
    # model.eval(): thực thi lệnh Python
    model.eval()
    # probs = ...: gán giá trị cho biến probs
    probs = []
    # for: vòng lặp — for ids, mask, beh, _ in make_loader(split):
    for ids, mask, beh, _ in make_loader(split):
        # probs.append(torch.sigmoid(model(ids.to(DEVICE), mask.to(DEVICE), beh.to(DEVICE)...: thực thi lệnh Python
        probs.append(torch.sigmoid(model(ids.to(DEVICE), mask.to(DEVICE), beh.to(DEVICE))).cpu().numpy())
    # return: nối các mảng numpy
    return np.concatenate(probs).astype(np.float32)

# best_macro_f1, best_state, wait, history = -np.inf, None, 0, []: thực thi lệnh Python
best_macro_f1, best_state, wait, history = -np.inf, None, 0, []
# start_time = ...: gán giá trị cho biến start time
start_time = time.time()
# for: vòng lặp — for epoch in range(1, CONFIG["max_epochs"] + 1):
for epoch in range(1, CONFIG["max_epochs"] + 1):
    # model.train(): thực thi lệnh Python
    model.train()
    # total_loss, seen = 0.0, 0: thực thi lệnh Python
    total_loss, seen = 0.0, 0
    # for: vòng lặp — for ids, mask, beh, labels in make_loader("train", shuffle=T
    for ids, mask, beh, labels in make_loader("train", shuffle=True):
        # ids, mask, beh, labels = ids.to(DEVICE), mask.to(DEVICE), beh.to(DEVICE), labels...: thực thi lệnh Python
        ids, mask, beh, labels = ids.to(DEVICE), mask.to(DEVICE), beh.to(DEVICE), labels.to(DEVICE)
        # optimizer.zero_grad(set_to_none=True): tạo tập hợp
        optimizer.zero_grad(set_to_none=True)
        # loss = ...: gán giá trị cho biến loss
        loss = criterion(model(ids, mask, beh), labels)
        # loss.backward(): thực thi lệnh Python
        loss.backward()
        # torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0): thực thi lệnh Python
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        # optimizer.step(): thực thi lệnh Python
        optimizer.step()
        # total_loss += float(loss.item()) * int(labels.size(0)): ép kiểu số thực
        total_loss += float(loss.item()) * int(labels.size(0))
        # seen += int(labels.size(0)): ép kiểu số nguyên
        seen += int(labels.size(0))
    # val_prob = ...: dự đoán nhãn/xác suất
    val_prob = predict("val")
    # row = ...: dự đoán nhãn/xác suất
    row = {"epoch": epoch, "train_loss": total_loss / max(seen, 1), **evaluate_predictions(y["val"], val_prob, "val", MODEL_VARIANT)}
    # history.append(row): thực thi lệnh Python
    history.append(row)
    # print(f"epoch {epoch:02d}: train_loss={row['train_loss']:.4f}, val_macro_f1={row...: in thông tin ra console
    print(f"epoch {epoch:02d}: train_loss={row['train_loss']:.4f}, val_macro_f1={row['macro_f1']:.4f}")
    # if: điều kiện — if row["macro_f1"] > best_macro_f1 + 1e-4:
    if row["macro_f1"] > best_macro_f1 + 1e-4:
        # best_macro_f1 = ...: gán giá trị cho biến best macro f1
        best_macro_f1 = row["macro_f1"]
        # best_state = ...: tạo dictionary
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        # wait = ...: gán giá trị cho biến wait
        wait = 0
    # else: nhánh còn lại của điều kiện
    else:
        # wait += 1: thực thi lệnh Python
        wait += 1
        # if: điều kiện — if wait >= CONFIG["patience"]:
        if wait >= CONFIG["patience"]:
            break
# if: điều kiện — if best_state is None:
if best_state is None:
    # raise RuntimeError("No best sequence state captured"): ném lỗi và dừng cell
    raise RuntimeError("No best sequence state captured")
# model.load_state_dict(best_state): tạo dictionary
model.load_state_dict(best_state)
# model_path = ...: gán giá trị cho biến model path
model_path = MODEL_DIR / f"{MODEL_VARIANT}.pth"
# torch.save({"model_variant": MODEL_VARIANT, "seed": SEED, "config": CONFIG, "beh...: tạo dictionary
torch.save({"model_variant": MODEL_VARIANT, "seed": SEED, "config": CONFIG, "behavior_cols": behavior_cols, "model_state_dict": best_state}, model_path)
# pd.DataFrame(history).to_csv(REPORT_TABLE_DIR / "phase5_cnn_bilstm_sequence_hist...: ghi DataFrame ra file CSV
pd.DataFrame(history).to_csv(REPORT_TABLE_DIR / "phase5_cnn_bilstm_sequence_history.csv", index=False)
# prob_map = ...: dự đoán nhãn/xác suất
prob_map = {split: predict(split) for split in ["train", "val", "test"]}
# metrics_df, metrics_path = write_metrics(prob_map, y, MODEL_VARIANT, "phase5_cnn...: thực thi lệnh Python
metrics_df, metrics_path = write_metrics(prob_map, y, MODEL_VARIANT, "phase5_cnn_bilstm_sequence_metrics.csv")
# with: context manager — with (MODEL_DIR / f"{MODEL_VARIANT}_metadata.json").open("w"
with (MODEL_DIR / f"{MODEL_VARIANT}_metadata.json").open("w", encoding="utf-8") as file:
    # json.dump({"model_variant": MODEL_VARIANT, "input_source": "token_sequence_plus_...: ghi dictionary ra JSON
    json.dump({"model_variant": MODEL_VARIANT, "input_source": "token_sequence_plus_behavioral_late_fusion", "model_path": str(model_path), "metrics_path": str(metrics_path), "train_seconds": round(time.time() - start_time, 3), "config": CONFIG}, file, indent=2)



=== seed=123 ===
epoch 01: train_loss=0.3254, val_macro_f1=0.9234
epoch 02: train_loss=0.1850, val_macro_f1=0.9305
epoch 03: train_loss=0.1452, val_macro_f1=0.9341
epoch 04: train_loss=0.1197, val_macro_f1=0.9320
epoch 05: train_loss=0.1002, val_macro_f1=0.9342
epoch 06: train_loss=0.0842, val_macro_f1=0.9252
epoch 07: train_loss=0.0711, val_macro_f1=0.9341
epoch 08: train_loss=0.0602, val_macro_f1=0.9255
epoch 09: train_loss=0.0511, val_macro_f1=0.9252
epoch 10: train_loss=0.0417, val_macro_f1=0.9319


,generated_at_utc,seed,split,model_variant,threshold,threshold_strategy,accuracy,macro_f1,precision_fake,recall_fake,...,support_real,support_fake,roc_auc,pr_auc,brier_score,tn,fp,fn,tp,probability_path
0,2026-06-10T09:13:23.071629+00:00,123,train,phase5_cnn_bilstm_sequence,0.5,default_0.5,0.981553,0.980892,0.982665,0.972073,...,17677,12246,0.996340,0.995210,0.015663,17467,210,342,11904,/content/drive/MyDrive/BaoMatCuoiKy/Fake_revie...
1,2026-06-10T09:13:23.102670+00:00,123,val,phase5_cnn_bilstm_sequence,0.5,default_0.5,0.937003,0.934247,0.948304,0.894817,...,3789,2624,0.972321,0.969385,0.053613,3661,128,276,2348,/content/drive/MyDrive/BaoMatCuoiKy/Fake_revie...
2,2026-06-10T09:13:23.135799+00:00,123,test,phase5_cnn_bilstm_sequence,0.5,default_0.5,0.933105,0.930265,0.939880,0.893674,...,3789,2624,0.970430,0.968267,0.055536,3639,150,279,2345,/content/drive/MyDrive/BaoMatCuoiKy/Fake_revie...


Saved metrics: /content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/reports/tables/seed_123/phase5_cnn_bilstm_sequence_metrics.csv
